<style>
.reveal h1, .reveal h2 { font-weight: 700; }
.reveal h1 { font-size: 2.15em; }
.reveal h2 { font-size: 1.55em; }
.reveal p, .reveal li { font-size: 0.90em; line-height: 1.35; }
.reveal table { font-size: 0.72em; }
.reveal .muted { color: #666; }
.reveal .accent { font-weight: 700; }
.reveal .big { font-size: 1.35em; font-weight: 700; }
.reveal .small { font-size: 0.72em; }
</style>

# Predicting Recognized Developmental Concern or Support

### A machine-learning proof of concept using the 2024 National Survey of Children's Health

**Victoria Nolasco, MD**  
AIM AI/ML Capstone Project

<br>

**Research question:** Can supervised machine learning identify patterns associated with *recognized developmental concern or support* among children ages 3–5?

## 1. Problem Framing

**Domain:** Healthcare  
**Task:** Supervised binary classification  
**Population:** Children ages 3–5 years

### Target
**Recognized Developmental Concern or Support**

Positive if the child had at least one qualifying indicator of:
- developmental / behavioral / learning / communication diagnosis
- developmental or behavioral treatment / service use
- early-intervention or special-education support
- another qualifying developmental-support indicator

> **Important:** The target is **not a diagnosis** and does not represent latent “true need.” It reflects *recognized concern or support* in survey data.

### Evaluation priorities
ROC-AUC • PR-AUC • F1 • Sensitivity • Precision • Specificity

## 2. Dataset and Analytic Cohort

### 2024 U.S. National Survey of Children's Health

<div style="display:flex; gap:28px; align-items:flex-start;">
<div style="flex:1;">

**Raw dataset**
- 51,375 observations
- 457 variables

**Analytic sample**
- Ages 3–5 only
- **N = 7,485**
- Age 3: 2,462
- Age 4: 2,489
- Age 5: 2,534

</div>
<div style="flex:1;">

**Outcome distribution**
- Positive: **2,030 (27.1%)**
- Negative: **5,455 (72.9%)**

**Initial feature set**
- **102 candidate predictors**
- 4 numerical
- 98 categorical

**Data quality**
- No exact duplicate rows
- Maximum missingness: **4.07%**
- No predictor >10% missing

</div>
</div>

## 3. Preprocessing, EDA, and Feature Engineering

### Reproducible preprocessing pipeline
1. **Age + outcome stratified 80/20 train-test split**
2. Numeric variables: median imputation + standardization
3. Categorical variables: most-frequent imputation + one-hot encoding
4. Transformations learned **inside the modeling pipeline**

### Applied EDA
- outcome prevalence and class imbalance
- age distribution and age-by-outcome strata
- missingness and coding checks
- subgroup performance by age and sex

### Feature engineering / selection
- constructed composite outcome
- excluded target-defining and downstream variables from predictors
- compared:
  - **Model A:** full eligible set
  - **Model B:** domain-informed reduction
  - **Model C:** SHAP-ranked data-driven reduction

### Dimensionality reduction
PCA showed substantial overlap between outcome groups, supporting supervised modeling rather than simple low-dimensional separation.

## 4. Model Implementation

### Models evaluated

| Model family | Role |
|---|---|
| Logistic Regression L1 / L2 | Interpretable linear baselines |
| Linear SVM | Margin-based classifier |
| Random Forest | Bagged tree ensemble |
| Gradient Boosting | Sequential tree ensemble |
| XGBoost | Regularized boosting |
| K-Nearest Neighbors | Distance-based baseline |

### Evaluation design
- **5-fold cross-validation** on training data
- Hyperparameter tuning for leading models
- Held-out test evaluation for Model A
- Calibration and threshold analysis
- Same Gradient Boosting architecture used for final A/B/C feature-set comparison

**Why multiple metrics?**  
With a 27.1% positive class, accuracy alone can mask poor case detection.

## 5. Model A: Full 102-Feature Model

### Strongest tuned model: Gradient Boosting

| Metric | Held-out test |
|---|---:|
| ROC-AUC | **0.784** |
| PR-AUC | **0.650** |
| Accuracy | **0.796** |
| Precision | **0.754** |
| Sensitivity | **0.369** |
| Specificity | **0.955** |

### Key interpretation

<div class="big">Good discrimination, but the default 0.50 threshold was too conservative for a screening-oriented use case.</div>

At 0.50, the model detected only about **37%** of positive cases.

Lower thresholds improved sensitivity, but increased the number of children flagged.  
**Threshold choice is therefore an operational decision, not a diagnostic cutoff.**

## 6. Feature Reduction Experiments

### Model B: Domain-informed
**38 features**

Selected using developmental-domain expertise and practical relevance:
- communication / language
- learning / attention
- regulation / social-emotional
- motor
- selected medical and routine variables

### Model C: Data-driven
**44 features**

Selected by:
1. fitting Gradient Boosting on Model A
2. ranking original predictors using aggregated SHAP importance
3. testing progressive feature subsets
4. identifying the performance plateau

### Convergence
**22 predictors were shared** by Models B and C.

This suggests meaningful overlap between domain expertise and data-driven importance, while also showing that some contextual predictors added predictive signal.

## 7. Final Model Comparison

<div style="display:flex; gap:24px; align-items:center;">
<div style="flex:1.15;">

| Model | Features | ROC-AUC | PR-AUC | F1 | Sens. |
|---|---:|---:|---:|---:|---:|
| A: Full | 102 | 0.782 | 0.662 | 0.510 | 0.381 |
| B: Domain | 38 | 0.766 | 0.643 | 0.500 | 0.373 |
| **C: Data-driven** | **44** | **0.785** | **0.664** | **0.518** | **0.390** |

</div>
<div style="flex:0.85;">

<svg viewBox="0 0 440 290" width="100%" xmlns="http://www.w3.org/2000/svg">
  <text x="10" y="24" font-size="18" font-weight="700">ROC-AUC</text>
  <line x1="95" y1="245" x2="410" y2="245" stroke="#999"/>
  <rect x="120" y="73" width="55" height="172" fill="#9aa0a6"/>
  <rect x="220" y="107" width="55" height="138" fill="#b8b8b8"/>
  <rect x="320" y="67" width="55" height="178" fill="#4f6d7a"/>
  <text x="128" y="65" font-size="15">.782</text>
  <text x="228" y="99" font-size="15">.766</text>
  <text x="328" y="59" font-size="15" font-weight="700">.785</text>
  <text x="135" y="268" font-size="14">A</text>
  <text x="235" y="268" font-size="14">B</text>
  <text x="335" y="268" font-size="14">C</text>
</svg>

</div>
</div>

### Takeaway
**Model C used fewer than half of the original predictors while matching or slightly exceeding Model A in exploratory cross-validation.**

<span class="small">Caution: Model C feature ranking was derived from the training set before subset CV; a future confirmatory analysis should use nested selection or external validation.</span>

## 8. Explainability and Fairness Audit

### SHAP: strongest predictive contributors
Age • clear expression • distractibility • sex • storytelling • focus • calming down • social interaction / regulation variables

**SHAP = predictive contribution, not causation.**

### Fairness check at a common 0.30 threshold

<div style="display:flex; gap:28px;">
<div style="flex:1;">

**Male**
- Sensitivity: **0.633**
- Specificity: 0.765
- Flagged: **36.3%**

</div>
<div style="flex:1;">

**Female**
- Sensitivity: **0.473**
- Specificity: 0.887
- Flagged: **19.2%**

</div>
</div>

### Ethical interpretation
The target reflects **recognized concern/support**, so historical differences in recognition and service access may already be embedded in the labels.

A model trained on those labels can reproduce those disparities.

**Fairness finding:** approximately **16 percentage-point sensitivity gap** by sex at the common threshold.

## 9. Conclusions and Next Steps

### What this capstone shows
- ML can identify meaningful patterns associated with recognized developmental concern/support.
- A **44-feature reduced model** retained the predictive performance of a 102-feature model.
- Domain-informed and data-driven feature selection showed substantial overlap.
- Threshold choice materially changes sensitivity, precision, and referral burden.
- Aggregate performance can conceal important subgroup disparities.

### What it does **not** show
- It is **not a diagnostic model**.
- It has not established latent developmental need.
- It has not been externally or prospectively validated.
- U.S. NSCH results should not be assumed to generalize to Philippine children.

### Next research phase
1. Validate prospectively using locally collected data
2. Use outcome labels closer to developmental need, not recognition alone
3. Pre-specify threshold and fairness criteria
4. Perform nested feature selection / external validation
5. Keep meaningful human oversight in any future decision-support workflow

**Bottom line:** predictive performance is only one part of responsible clinical translation.

Presenter note: Emphasize that the project is deliberately framed as proof-of-concept decision-support research, not diagnosis.